# 🔬 PAS Silver — Deep Dive Exploration

**Project**: Global Loom  
**Lakehouse**: `The_Global_Loom`  
**Purpose**: Investigate the key unknowns before star schema design.

### What we're investigating:
1. **vwPolicy** — Why is it 20M rows? Dedup check.
2. **Invoice ↔ Transaction chain** — FK integrity, coverage, orphans
3. **Transaction ↔ Detail chain** — 1:many ratio, orphans
4. **Product coverage** — How much data has product info?
5. **Invoice as primary fact** — Does CFInvoice have everything we need?

> ⚠️ Some queries touch 471M+ rows — expect 2-5 min runtimes.

In [ ]:
# ============================================================
# Setup — reuse same config as exploration notebook
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")

print("✅ Ready")

---
## 1️⃣ vwPolicy — Why 20M rows?

This is the #1 concern. 20M is too large for a dimension. We need to understand:
- How many **unique policies** exist?
- Is it duplicated across **data source instances** (Eclipse, WIBS, etc.)?
- Are there **multiple versions** per policy (SCD/history)?

In [ ]:
%%sql
-- Q1a: Overall uniqueness check
SELECT
    COUNT(*)                              AS TotalRows,
    COUNT(DISTINCT PolicyId)              AS UniquePolicyIds,
    COUNT(DISTINCT PolicyKey)             AS UniquePolicyKeys,
    COUNT(DISTINCT DataSourceInstanceId)  AS DataSources,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT PolicyId), 1) AS AvgRowsPerPolicy
FROM rpt.vwPolicy

In [ ]:
%%sql
-- Q1b: Row count by DataSourceInstance — is one source dominating?
SELECT
    p.DataSourceInstanceId,
    d.DataSourceInstanceName,
    COUNT(*)                    AS RowCount,
    COUNT(DISTINCT p.PolicyId)  AS UniquePolicies
FROM rpt.vwPolicy p
LEFT JOIN rpt.vwDataSourceInstance d
    ON p.DataSourceInstanceId = d.DataSourceInstanceId
GROUP BY p.DataSourceInstanceId, d.DataSourceInstanceName
ORDER BY RowCount DESC

In [ ]:
%%sql
-- Q1c: Are there duplicate PolicyIds WITHIN the same DataSourceInstance?
-- (If yes → versioning/SCD. If no → cross-source duplication only.)
SELECT
    DataSourceInstanceId,
    COUNT(*) AS DuplicatePolicies
FROM (
    SELECT DataSourceInstanceId, PolicyId, COUNT(*) AS cnt
    FROM rpt.vwPolicy
    GROUP BY DataSourceInstanceId, PolicyId
    HAVING COUNT(*) > 1
)
GROUP BY DataSourceInstanceId
ORDER BY DuplicatePolicies DESC

In [ ]:
%%sql
-- Q1d: Can PolicyKey deduplicate? (PolicyKey is often DSI-scoped)
SELECT
    COUNT(*)                    AS TotalRows,
    COUNT(DISTINCT PolicyKey)   AS UniqueKeys,
    CASE
        WHEN COUNT(*) = COUNT(DISTINCT PolicyKey) THEN '✅ PolicyKey IS unique'
        ELSE '❌ PolicyKey has duplicates'
    END AS Verdict
FROM rpt.vwPolicy

---
## 2️⃣ Invoice ↔ Transaction Chain

If we pivot around CFInvoice, we need to know:
- Do all invoices link to a valid transaction?
- How many transactions have invoices vs. don't?
- What % of the financial universe does CFInvoice cover?

In [ ]:
%%sql
-- Q2a: Invoice FK integrity — do all invoices have a matching transaction?
SELECT
    COUNT(*)                                                           AS TotalInvoices,
    SUM(CASE WHEN t.TransactionId IS NOT NULL THEN 1 ELSE 0 END)       AS MatchedToTransaction,
    SUM(CASE WHEN t.TransactionId IS NULL THEN 1 ELSE 0 END)           AS OrphanInvoices,
    ROUND(SUM(CASE WHEN t.TransactionId IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS MatchPct
FROM rpt.vwCFInvoice i
LEFT JOIN rpt.vwTransaction t
    ON i.TransactionId = t.TransactionId

In [ ]:
%%sql
-- Q2b: How many transactions have invoices? (Coverage)
SELECT
    COUNT(*)                                                           AS TotalTransactions,
    SUM(CASE WHEN i.TransactionId IS NOT NULL THEN 1 ELSE 0 END)       AS HasInvoice,
    SUM(CASE WHEN i.TransactionId IS NULL THEN 1 ELSE 0 END)           AS NoInvoice,
    ROUND(SUM(CASE WHEN i.TransactionId IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS CoveragePct
FROM rpt.vwTransaction t
LEFT JOIN (
    SELECT DISTINCT TransactionId FROM rpt.vwCFInvoice
) i ON t.TransactionId = i.TransactionId

In [ ]:
%%sql
-- Q2c: Invoice-to-Transaction ratio (how many invoices per transaction?)
SELECT
    CASE
        WHEN cnt = 1 THEN '1 invoice'
        WHEN cnt BETWEEN 2 AND 5 THEN '2-5 invoices'
        WHEN cnt BETWEEN 6 AND 10 THEN '6-10 invoices'
        WHEN cnt BETWEEN 11 AND 50 THEN '11-50 invoices'
        ELSE '50+ invoices'
    END AS InvoicesPerTransaction,
    COUNT(*) AS TransactionCount
FROM (
    SELECT TransactionId, COUNT(*) AS cnt
    FROM rpt.vwCFInvoice
    GROUP BY TransactionId
)
GROUP BY
    CASE
        WHEN cnt = 1 THEN '1 invoice'
        WHEN cnt BETWEEN 2 AND 5 THEN '2-5 invoices'
        WHEN cnt BETWEEN 6 AND 10 THEN '6-10 invoices'
        WHEN cnt BETWEEN 11 AND 50 THEN '11-50 invoices'
        ELSE '50+ invoices'
    END
ORDER BY TransactionCount DESC

In [ ]:
%%sql
-- Q2d: Can we get Product info through the transaction chain?
-- Join Invoice → Transaction → check Product fields
SELECT
    COUNT(*)                                                                  AS TotalInvoices,
    SUM(CASE WHEN t.ProductId IS NOT NULL AND t.ProductId != -1 THEN 1 ELSE 0 END) AS HasProduct,
    SUM(CASE WHEN t.GlobalProductClass IS NOT NULL THEN 1 ELSE 0 END)          AS HasProductClass,
    ROUND(
        SUM(CASE WHEN t.ProductId IS NOT NULL AND t.ProductId != -1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS ProductCoveragePct
FROM rpt.vwCFInvoice i
LEFT JOIN rpt.vwTransaction t
    ON i.TransactionId = t.TransactionId

In [ ]:
%%sql
-- Q2e: What dimensions can Invoice pick up through Transaction?
SELECT
    COUNT(*) AS TotalInvoices,
    SUM(CASE WHEN t.PolicyId IS NOT NULL AND t.PolicyId != -1 THEN 1 ELSE 0 END)                     AS HasPolicy,
    SUM(CASE WHEN t.ProductId IS NOT NULL AND t.ProductId != -1 THEN 1 ELSE 0 END)                   AS HasProduct,
    SUM(CASE WHEN t.GlobalFinancialGeographyId IS NOT NULL AND t.GlobalFinancialGeographyId != -1 THEN 1 ELSE 0 END) AS HasGeography,
    SUM(CASE WHEN t.GlobalFinancialSegmentId IS NOT NULL AND t.GlobalFinancialSegmentId != -1 THEN 1 ELSE 0 END)     AS HasSegment
FROM rpt.vwCFInvoice i
LEFT JOIN rpt.vwTransaction t
    ON i.TransactionId = t.TransactionId

---
## 3️⃣ Transaction ↔ Detail Chain

The Detail table (471M) hangs off the Transaction header (85M). We need to verify:
- FK integrity between them
- The 1:many ratio
- Orphans in either direction

In [ ]:
%%sql
-- Q3a: Basic counts and distinct TransactionIds in each table
SELECT 'vwTransaction'          AS TableName, COUNT(*) AS TotalRows, COUNT(DISTINCT TransactionId) AS UniqueTransactionIds FROM rpt.vwTransaction
UNION ALL
SELECT 'vwTransactionDetailUSD' AS TableName, COUNT(*) AS TotalRows, COUNT(DISTINCT TransactionId) AS UniqueTransactionIds FROM rpt.vwTransactionDetailUSD
UNION ALL
SELECT 'vwTransactionSummaryUSD' AS TableName, COUNT(*) AS TotalRows, COUNT(DISTINCT PolicyId) AS UniquePolicyIds FROM rpt.vwTransactionSummaryUSD

In [ ]:
%%sql
-- Q3b: Orphan details — rows in Detail that have NO matching header
-- ⚠️ May take 2-5 min on 471M rows
SELECT COUNT(*) AS OrphanDetailRows
FROM rpt.vwTransactionDetailUSD d
LEFT ANTI JOIN rpt.vwTransaction t
    ON d.TransactionId = t.TransactionId

In [ ]:
%%sql
-- Q3c: Orphan headers — transactions with NO detail rows
SELECT COUNT(*) AS HeadersWithoutDetails
FROM rpt.vwTransaction t
LEFT ANTI JOIN rpt.vwTransactionDetailUSD d
    ON t.TransactionId = d.TransactionId

In [ ]:
%%sql
-- Q3d: Detail-to-Header ratio distribution
SELECT
    CASE
        WHEN cnt = 1 THEN '1 detail'
        WHEN cnt BETWEEN 2 AND 5 THEN '2-5 details'
        WHEN cnt BETWEEN 6 AND 10 THEN '6-10 details'
        WHEN cnt BETWEEN 11 AND 50 THEN '11-50 details'
        ELSE '50+ details'
    END AS DetailsPerTransaction,
    COUNT(*) AS TransactionCount,
    SUM(cnt) AS TotalDetailRows
FROM (
    SELECT TransactionId, COUNT(*) AS cnt
    FROM rpt.vwTransactionDetailUSD
    GROUP BY TransactionId
)
GROUP BY
    CASE
        WHEN cnt = 1 THEN '1 detail'
        WHEN cnt BETWEEN 2 AND 5 THEN '2-5 details'
        WHEN cnt BETWEEN 6 AND 10 THEN '6-10 details'
        WHEN cnt BETWEEN 11 AND 50 THEN '11-50 details'
        ELSE '50+ details'
    END
ORDER BY TransactionCount DESC

---
## 4️⃣ Product Exploration

Product info lives on vwTransaction (not on Detail or Invoice). Let's understand the product hierarchy and coverage.

In [ ]:
%%sql
-- Q4a: Product table overview
SELECT
    COUNT(*)                         AS TotalProducts,
    COUNT(DISTINCT ProductId)        AS UniqueProductIds,
    COUNT(DISTINCT ProductKey)       AS UniqueProductKeys,
    COUNT(DISTINCT GlobalProductClassId)  AS UniqueProductClasses,
    COUNT(DISTINCT GlobalProductLineId)   AS UniqueProductLines,
    COUNT(DISTINCT GlobalProductId)       AS UniqueGlobalProducts,
    COUNT(DISTINCT DataSourceInstanceId)  AS DataSources
FROM rpt.vwProduct

In [ ]:
%%sql
-- Q4b: Product hierarchy — Class → Line → Product
SELECT
    GlobalProductClass,
    GlobalProductLine,
    COUNT(DISTINCT GlobalProductId) AS UniqueProducts,
    COUNT(DISTINCT ProductId)       AS LocalProducts
FROM rpt.vwProduct
WHERE GlobalProductClass IS NOT NULL
GROUP BY GlobalProductClass, GlobalProductLine
ORDER BY UniqueProducts DESC
LIMIT 30

In [ ]:
%%sql
-- Q4c: Product coverage on transactions
SELECT
    COUNT(*)                                                                   AS TotalTransactions,
    SUM(CASE WHEN ProductId IS NOT NULL AND ProductId != -1 THEN 1 ELSE 0 END) AS HasProductId,
    SUM(CASE WHEN GlobalProductClass IS NOT NULL THEN 1 ELSE 0 END)            AS HasProductClass,
    SUM(CASE WHEN GlobalProductClass IS NULL THEN 1 ELSE 0 END)                AS MissingProductClass,
    ROUND(
        SUM(CASE WHEN ProductId IS NOT NULL AND ProductId != -1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS ProductCoveragePct
FROM rpt.vwTransaction

In [ ]:
%%sql
-- Q4d: Do ProductIds on transactions match the Product table?
SELECT
    COUNT(*)                                                         AS TotalTransactions,
    SUM(CASE WHEN p.ProductId IS NOT NULL THEN 1 ELSE 0 END)         AS MatchedToProduct,
    SUM(CASE WHEN t.ProductId != -1 AND p.ProductId IS NULL THEN 1 ELSE 0 END) AS OrphanProductIds
FROM rpt.vwTransaction t
LEFT JOIN rpt.vwProduct p
    ON t.ProductId = p.ProductId

---
## 5️⃣ CFInvoice as Primary Fact — Feasibility Check

If we pivot around CFInvoice, what does it give us natively vs. what needs enrichment?

In [ ]:
%%sql
-- Q5a: CFInvoice — basic shape and key distributions
SELECT
    COUNT(*)                               AS TotalInvoices,
    COUNT(DISTINCT TransactionId)          AS UniqueTransactionIds,
    COUNT(DISTINCT PartyId)                AS UniqueParties,
    COUNT(DISTINCT CustomerNumber)         AS UniqueCustomers,
    COUNT(DISTINCT DocumentNumber)         AS UniqueDocuments,
    COUNT(DISTINCT DataSourceInstanceId)   AS DataSources
FROM rpt.vwCFInvoice

In [ ]:
%%sql
-- Q5b: DocumentType distribution — what kinds of invoices?
SELECT DocumentType, COUNT(*) AS Cnt
FROM rpt.vwCFInvoice
GROUP BY DocumentType
ORDER BY Cnt DESC

In [ ]:
%%sql
-- Q5c: GlobalPartyRole distribution on invoices
SELECT GlobalPartyRole, COUNT(*) AS Cnt
FROM rpt.vwCFInvoice
GROUP BY GlobalPartyRole
ORDER BY Cnt DESC

In [ ]:
%%sql
-- Q5d: Financial summary — DocumentAmount & CorporateAmount
SELECT
    COUNT(*)                                                               AS TotalRows,
    SUM(CASE WHEN DocumentAmount IS NOT NULL THEN 1 ELSE 0 END)            AS HasDocAmount,
    SUM(CASE WHEN CorporateAmount IS NOT NULL THEN 1 ELSE 0 END)           AS HasCorpAmount,
    COUNT(DISTINCT Currency)                                               AS UniqueCurrencies,
    ROUND(SUM(DocumentAmount), 2)                                          AS TotalDocAmount,
    ROUND(SUM(CorporateAmount), 2)                                         AS TotalCorpAmount
FROM rpt.vwCFInvoice

In [ ]:
%%sql
-- Q5e: Date range on invoices
SELECT
    MIN(DocumentDate)       AS EarliestInvoice,
    MAX(DocumentDate)       AS LatestInvoice,
    MIN(DocumentDueDate)    AS EarliestDueDate,
    MAX(DocumentDueDate)    AS LatestDueDate,
    COUNT(DISTINCT DocumentDate) AS UniqueDates
FROM rpt.vwCFInvoice

In [ ]:
%%sql
-- Q5f: CFParty coverage — do all invoice CustomerNumbers match CFParty?
SELECT
    COUNT(*)                                                         AS TotalInvoices,
    SUM(CASE WHEN cp.CustomerNumber IS NOT NULL THEN 1 ELSE 0 END)   AS MatchedToCFParty,
    SUM(CASE WHEN cp.CustomerNumber IS NULL THEN 1 ELSE 0 END)       AS UnmatchedInvoices
FROM rpt.vwCFInvoice i
LEFT JOIN rpt.vwCFParty cp
    ON i.CustomerNumber = cp.CustomerNumber
    AND i.DataSourceInstanceId = cp.DataSourceInstanceId

---
## ✅ Summary

After running all queries, share the outputs and we'll decide:
1. Is `vwPolicy` duplicated by DataSource or versioned? → Drives dedup strategy
2. Does CFInvoice reliably link to Transaction? → Drives fact table choice
3. How complete is the Transaction → Detail chain? → Data integrity
4. How much Product coverage do we have? → Dimension design
5. Is CFInvoice viable as a primary fact? → Architecture decision